PePy

Author: Julia K. Varga <jvarga92@gmail.com>  
License: BSD 3 clause  
Code Repository: https://github.com/gezmi/pepy

# Command Line Interface

PePy includes a CLI tool for scoring structure files from the terminal.
After installing (`pip install -e .`), the `pepy` command is available.

This mirrors the functionality of the original `fast_avg_plddt_window_ipae_iptm.py` script, but uses the refactored PePy package under the hood.

## Help

In [ ]:
!python -m pepy.cli --help

## Single File

Score a single structure. By default, only the interface is calculated (no confidence metrics):

In [ ]:
!python -m pepy.cli \
    -i ../../pepy/tests/data/1ycr_af2_55d19_unrelaxed_rank_001_alphafold2_multimer_v3_model_1_seed_000.pdb \
    -o /tmp/pepy_single.tsv

In [ ]:
import pandas as pd
pd.read_csv('/tmp/pepy_single.tsv', sep='\t')

## With Confidence Metrics

Add `-j` (or `--confidence`) to also load and report iPTM, pTM, iPAE, and combined confidence.
PePy auto-discovers the confidence file (JSON for AF2/AF3, NPZ for ChAI) based on the structure filename:

In [ ]:
!python -m pepy.cli \
    -i ../../pepy/tests/data/1ycr_af2_55d19_unrelaxed_rank_001_alphafold2_multimer_v3_model_1_seed_000.pdb \
    -o /tmp/pepy_confidence.tsv \
    -j

In [ ]:
pd.read_csv('/tmp/pepy_confidence.tsv', sep='\t')

## Batch Processing

Process multiple files at once. Three input modes:

| Flag | Description |
|------|-------------|
| `-i` | Single file |
| `-l` | Text file with one path per line |
| `-g` | Glob pattern (e.g. `"predictions/*.pdb"`) |

### Using a glob pattern

In [ ]:
!python -m pepy.cli \
    -g "../../pepy/tests/data/*.pdb" \
    -o /tmp/pepy_batch.tsv \
    -j

In [ ]:
pd.read_csv('/tmp/pepy_batch.tsv', sep='\t')

### Using a file list

Create a text file with one structure path per line, then pass it with `-l`:

In [ ]:
# Create a file list
with open('/tmp/pepy_files.txt', 'w') as f:
    f.write('../../pepy/tests/data/1ycr_af2_55d19_unrelaxed_rank_001_alphafold2_multimer_v3_model_1_seed_000.pdb\n')
    f.write('../../pepy/tests/data/pred.model_idx_0.pdb\n')

In [ ]:
!python -m pepy.cli \
    -l /tmp/pepy_files.txt \
    -o /tmp/pepy_from_list.tsv \
    -j

In [ ]:
pd.read_csv('/tmp/pepy_from_list.tsv', sep='\t')

## Specifying Chains

By default, the shortest chain is assigned as binder. Use `-b` and `-r` for explicit assignment.
Both accept comma-separated chain IDs for multi-chain binder/receptor:

In [ ]:
!python -m pepy.cli \
    -i ../../pepy/tests/data/1ycr_af2_55d19_unrelaxed_rank_001_alphafold2_multimer_v3_model_1_seed_000.pdb \
    -o /tmp/pepy_chains.tsv \
    -b B -r A \
    -j

In [ ]:
pd.read_csv('/tmp/pepy_chains.tsv', sep='\t')

## Interface Parameters

Control the two-stage interface calculation:

| Flag | Default | Description |
|------|---------|-------------|
| `--cb-cutoff` | 8.0 | CB prefilter distance (Å). Use `-1` to skip. |
| `--all-atom-cutoff` | 4.0 | All-atom refinement distance (Å). Use `-1` to skip. |
| `--min-interface` | 1 | Minimum binder residues to count as interface |
| `-d` | off | Drop binder residues with pLDDT below threshold |
| `--confidence-threshold` | 50.0 | pLDDT threshold when `-d` is used |

In [ ]:
# Stricter cutoffs + drop low confidence residues
!python -m pepy.cli \
    -i ../../pepy/tests/data/1ycr_af2_55d19_unrelaxed_rank_001_alphafold2_multimer_v3_model_1_seed_000.pdb \
    -o /tmp/pepy_strict.tsv \
    --cb-cutoff 6.0 --all-atom-cutoff 3.0 \
    -d --confidence-threshold 70.0 \
    -j

In [ ]:
pd.read_csv('/tmp/pepy_strict.tsv', sep='\t')

## Parallel Processing

Use `-c` to process multiple files in parallel (requires `joblib`; install with `pip install pepy[parallel]`):

```bash
# Use all available cores
pepy -g "predictions/*.pdb" -o results.tsv -j -c -1

# Use 4 cores
pepy -g "predictions/*.pdb" -o results.tsv -j -c 4
```

## Output Format

The output is a tab-separated file. Columns depend on whether `-j` is used:

**Always present:**

| Column | Description |
|--------|-------------|
| `file` | Input filename |
| `binder` | Binder chain(s) |
| `receptor` | Receptor chain(s) |
| `n_binder_res` | Number of binder interface residues |
| `n_receptor_res` | Number of receptor interface residues |
| `avg_plddt` | Mean pLDDT of binder interface CA atoms |
| `max_plddt` | Max pLDDT of binder interface CA atoms |

**With `-j` (confidence):**

| Column | Description |
|--------|-------------|
| `iptm` | Interface predicted TM-score |
| `ptm` | Predicted TM-score |
| `ipae` | Median PAE between binder–receptor interface |
| `min_ipae` | Min PAE between binder–receptor interface |
| `confidence` | Combined score (0.8 × iPTM + 0.2 × pTM) |

If a file fails to process, an `error` column will appear for that row.

## Comparison with Original Script

If you were using `fast_avg_plddt_window_ipae_iptm.py`, here's the mapping:

| Old flag | New flag | Notes |
|----------|----------|-------|
| `-i` | `-i` | Same |
| `-l` | `-l` | Same |
| — | `-g` | New: glob patterns |
| `-o` | `-o` | Same |
| `-p` | `-b` | Renamed: peptide → binder |
| `-r` | `-r` | Same |
| `-b` (cb cutoff) | `--cb-cutoff` | Now a long flag |
| `-j` | `-j` | Same |
| `-d` | `-d` | Same |
| `-m` | `--min-interface` | Now a long flag |
| `-c` | `-c` | Same |
| `-w`, `-e` | — | Removed (windowing removed entirely) |
| `-f` (focus) | — | Removed |
| `--convert_cif` | — | Automatic now (AF3 CIF fixing is built-in) |